# LungRay: ResNet-50 Consensus Backbone Training 

Tiền điều kiện (làm trước khi chạy):
1. Settings: Accelerator = GPU P100 (hoặc 2x T4), Internet = ON.
2. Add Input: `lungray-nih-512`, `lungray-covidx-512`, `tuberculosis-tb-chest-xray-dataset`.
3. Điền `REPO_COMMIT` (SHA lấy bằng `git rev-parse HEAD` trên máy) trước khi chạy `RUN_MODE = "full"`.
4. Quy trình: chạy `RUN_MODE = "smoke"` trước (đo tốc độ, kiểm tra pipeline), rồi đổi sang `"full"`.
5. Với run dài, dùng Save Version -> Save & Run All để /kaggle/working được lưu sau khi phiên kết thúc.

Đầu ra: `best_resnet50.pth` (nguyên định dạng script xuất), `best_resnet50_weights_only.pth`,
`optimal_thresholds_resnet50.json` (script ghi vào configs/, cell 8 sao chép ra /kaggle/working; không ghi đè ngưỡng DenseNet).

In [1]:
# Cell 1: Cấu hình
from pathlib import Path
import os, sys, json, time, subprocess, hashlib, random, shlex

REPO_URL = "https://github.com/philipsgn/LungRay.git"
REPO_COMMIT = "4f1e1ec"        # Ghim commit fix chuẩn
RUN_MODE = "full"              # Đổi từ smoke sang full
SEED = 42

BACKBONE = "resnet50"
EPOCHS = 15
BATCH_SIZE = 20                # 12:5:3 = đúng 60:25:15
LR = 1e-4
WEIGHT_DECAY = 1e-4
MAX_BATCHES = 0                # 0 = Chạy 100% full dataset (6,548 batches/epoch)
MAX_VAL_BATCHES = 0            # 0 = Đánh giá validation đầy đủ
HEAD_C_CRITERION = "bce"
RESUME = False
TIME_BUDGET_HOURS = 7.5

WORK = Path("/kaggle/working")
REPO = WORK / "LungRay"
CKPT_NAME = "best_resnet50.pth"
THRESHOLDS_OUT = WORK / "optimal_thresholds_resnet50.json"
THRESHOLDS_SRC = REPO / "configs" / "optimal_thresholds_resnet50.json"
TIMING_JSON = WORK / "smoke_timing.json"

assert RUN_MODE in ("smoke", "full")
for name, share in (("NIH", 0.60), ("COVIDx", 0.25), ("TB", 0.15)):
    n = share * BATCH_SIZE
    flag = "OK" if abs(n - round(n)) < 1e-9 else "KHONG CHIA HET -> sampler phai lam tron"
    print(f"{name}: {n:.1f} mau/batch [{flag}]")

NIH: 12.0 mau/batch [OK]
COVIDx: 5.0 mau/batch [OK]
TB: 3.0 mau/batch [OK]


In [2]:
# Cell 2: Môi trường, seed, GPU
import numpy as np
import torch

os.environ.update({
    "NO_ALBUMENTATIONS_UPDATE": "1",
    "LUNGRAY_NUM_WORKERS": "4",
    "OMP_NUM_THREADS": "1",
    "MKL_NUM_THREADS": "1",
    "PYTHONHASHSEED": str(SEED),
})
random.seed(SEED)
np.random.seed(SEED)
torch.manual_seed(SEED)
torch.cuda.manual_seed_all(SEED)
torch.backends.cudnn.deterministic = True
torch.backends.cudnn.benchmark = False

print("PyTorch:", torch.__version__)
assert torch.cuda.is_available(), "Chua bat GPU: Settings -> Accelerator -> GPU"
vram_gb = torch.cuda.get_device_properties(0).total_memory / 1024 ** 3
print("GPU:", torch.cuda.get_device_name(0), f"| VRAM {vram_gb:.1f} GB")
if vram_gb < 14:
    print("[!] VRAM < 14 GB: nen ha BATCH_SIZE xuong 12-16")

PyTorch: 2.11.0+cu130
GPU: Tesla T4 | VRAM 14.6 GB


In [3]:
# Cell 3: Helper chạy lệnh (stream log, báo lỗi rõ ràng)
def run(cmd, check=True, cwd=None):
    cmd = [str(c) for c in cmd]
    print("$", " ".join(shlex.quote(c) for c in cmd), flush=True)
    t0 = time.time()
    proc = subprocess.Popen(cmd, cwd=cwd, stdout=subprocess.PIPE,
                            stderr=subprocess.STDOUT, text=True, bufsize=1)
    lines = []
    for line in proc.stdout:
        print(line, end="", flush=True)
        lines.append(line)
    rc = proc.wait()
    dt = time.time() - t0
    if check and rc != 0:
        raise RuntimeError(f"Lenh that bai (rc={rc}). 20 dong cuoi:\n" + "".join(lines[-20:]))
    return rc, dt, "".join(lines)

run([sys.executable, "-m", "pip", "install", "-q",
     "albumentations", "timm", "opencv-python-headless", "scikit-learn", "pandas", "matplotlib"])

$ /usr/bin/python3 -m pip install -q albumentations timm opencv-python-headless scikit-learn pandas matplotlib


(0, 2.7399160861968994, '')

In [4]:
# Cell 4: Clone repo (idempotent, ghim commit)
if not REPO.exists():
    run(["git", "clone", REPO_URL, REPO])
if REPO_COMMIT:
    run(["git", "-C", REPO, "fetch", "--all", "-q"])
    run(["git", "-C", REPO, "checkout", "-q", REPO_COMMIT])
elif RUN_MODE == "full":
    raise RuntimeError("RUN_MODE='full' yeu cau REPO_COMMIT de ket qua tai lap duoc.")
else:
    print("[!] REPO_COMMIT trong: chi chap nhan cho smoke test.")
run(["git", "-C", REPO, "rev-parse", "HEAD"])
sys.path.insert(0, str(REPO))
os.chdir(REPO)
print("[OK] Workspace:", os.getcwd())

$ git clone https://github.com/philipsgn/LungRay.git /kaggle/working/LungRay
Cloning into '/kaggle/working/LungRay'...
$ git -C /kaggle/working/LungRay fetch --all -q
$ git -C /kaggle/working/LungRay checkout -q 4f1e1ec
$ git -C /kaggle/working/LungRay rev-parse HEAD
4f1e1ec753b5c2d31e9aa5cd49c5833afa2ea2a0
[OK] Workspace: /kaggle/working/LungRay


In [5]:
# Cell 5: Preflight (dữ liệu, script, cờ CLI)
INPUT = Path("/kaggle/input")

def find_dataset(slug_hints, required):
    for root, dirs, _ in os.walk(INPUT):
        rel = Path(root).relative_to(INPUT).parts
        if len(rel) > 5:
            dirs[:] = []
            continue
        if any(h in Path(root).name.lower() for h in slug_hints):
            rootp = Path(root)
            if all(next(rootp.rglob(r), None) is not None for r in required):
                return rootp
    return None

checks = {
    "NIH 512": find_dataset(["lungray-nih-512", "nih-512", "nih_512"], ["patient_splits.csv"]),
    "COVIDx 512": find_dataset(["lungray-covidx-512", "covidx-512", "covidx"], ["train.txt", "test.txt"]),
    "TB": find_dataset(["tuberculosis"], ["Normal", "Tuberculosis"]),
}
for k, v in checks.items():
    print(f"{k:12s}: {v if v else 'THIEU'}")
missing_ds = [k for k, v in checks.items() if v is None]
assert not missing_ds, f"Thieu dataset: {missing_ds}. Add Input dung slug roi chay lai."

SCRIPT = REPO / "kaggle" / "train_resnet50_kaggle.py"
assert SCRIPT.exists(), f"Khong thay {SCRIPT}"
_, _, help_txt = run([sys.executable, SCRIPT, "--help"])

REQUIRED = ["--backbone", "--epochs", "--batch-size", "--lr", "--weight-decay",
            "--head-a-criterion", "--head-c-criterion", "--asl-gamma-neg", "--asl-gamma-pos",
            "--asl-clip", "--max-batches", "--max-val-batches", "--save-name",
            "--resume", "--no-calibrate"]
miss_req = [f for f in REQUIRED if f not in help_txt]
assert not miss_req, f"Script thieu co bat buoc: {miss_req}"

# Script khong co --seed: seed trong tien trinh con bang wrapper (runpy giu nguyen __file__/__main__).
SEED_WRAPPER = (
    "import sys, runpy, random\n"
    f"S = {SEED}\n"
    "random.seed(S)\n"
    "import numpy as np, torch\n"
    "np.random.seed(S); torch.manual_seed(S); torch.cuda.manual_seed_all(S)\n"
    "torch.backends.cudnn.deterministic = True; torch.backends.cudnn.benchmark = False\n"
    "sys.argv = sys.argv[1:]\n"
    "runpy.run_path(sys.argv[0], run_name='__main__')\n"
)

def build_cmd(epochs, max_batches, max_val_batches, save_name, smoke=False, resume=False):
    cmd = [sys.executable, "-c", SEED_WRAPPER, SCRIPT,
           "--backbone", BACKBONE, "--epochs", epochs, "--batch-size", BATCH_SIZE,
           "--lr", LR, "--weight-decay", WEIGHT_DECAY,
           "--head-a-criterion", "asl", "--head-c-criterion", HEAD_C_CRITERION,
           "--asl-gamma-neg", 4.0, "--asl-gamma-pos", 1.0, "--asl-clip", 0.05,
           "--max-batches", max_batches, "--max-val-batches", max_val_batches,
           "--save-name", save_name]
    if smoke:
        cmd += ["--no-calibrate"]      # smoke khong duoc ghi file nguong
    if resume:
        cmd += ["--resume"]
    return cmd

NIH 512     : /kaggle/input/datasets/phanphat/lungray-nih-512
COVIDx 512  : /kaggle/input/datasets/phanphat/lungray-covidx-512
TB          : /kaggle/input/datasets/tawsifurrahman/tuberculosis-tb-chest-xray-dataset
$ /usr/bin/python3 /kaggle/working/LungRay/kaggle/train_resnet50_kaggle.py --help
usage: train_resnet50_kaggle.py [-h] [--epochs EPOCHS]
                                [--batch-size BATCH_SIZE] [--lr LR]
                                [--weight-decay WEIGHT_DECAY] [--seed SEED]
                                [--backbone BACKBONE]
                                [--head-a-criterion {bce,focal,asl}]
                                [--head-c-criterion {bce,focal,asl}]
                                [--asl-gamma-neg ASL_GAMMA_NEG]
                                [--asl-gamma-pos ASL_GAMMA_POS]
                                [--asl-clip ASL_CLIP]
                                [--max-batches MAX_BATCHES]
                                [--max-val-batches MAX_VAL_BATCHES]
   

In [6]:
# Cell 6: Smoke test + đo tốc độ (chỉ RUN_MODE="smoke")
def cleanup_smoke(t_start):
    # Xoa moi checkpoint .pth do smoke tao ra (mtime >= t_start), gom ca last_resnet50.pth.
    for base in (REPO, WORK):
        for p in base.rglob("*.pth"):
            if p.is_file() and p.stat().st_mtime >= t_start:
                p.unlink()
                print("removed:", p)
    for p in WORK.glob("smoke_*"):
        if p.is_file():
            p.unlink()

if RUN_MODE == "smoke":
    t_start = time.time()
    t = {}
    for n in (20, 60):
        _, dt, _ = run(build_cmd(1, n, 4, "smoke_resnet50.pth", smoke=True))
        t[n] = dt
    spb = (t[60] - t[20]) / 40.0
    if spb <= 0:
        print("[!] Do toc do khong tin cay (spb<=0). Chay lai smoke.")
    else:
        affordable = int(TIME_BUDGET_HOURS * 3600 / (spb * EPOCHS) * 0.85)
        rec = max(100, (affordable // 100) * 100)
        TIMING_JSON.write_text(json.dumps({"sec_per_batch": spb, "recommended_max_batches": rec,
                                           "epochs": EPOCHS, "batch_size": BATCH_SIZE}))
        print(f"sec/batch = {spb:.3f} | ngan sach {TIME_BUDGET_HOURS} h, {EPOCHS} epoch "
              f"-> MAX_BATCHES khuyen nghi = {rec}")
        print("Neu so nay < 2000: giam EPOCHS hoac chap nhan huan luyen ngan hon, KHONG dung 600.")
    cleanup_smoke(t_start)   # tranh --resume nham vao checkpoint cua smoke
else:
    print("Bo qua smoke (RUN_MODE='full').")

Bo qua smoke (RUN_MODE='full').


In [7]:
# Cell 7: Huấn luyện thật (chỉ RUN_MODE="full")
if RUN_MODE == "full":
    timing = json.loads(TIMING_JSON.read_text()) if TIMING_JSON.exists() else None
    mb = MAX_BATCHES
    if mb is None:
        assert timing, "MAX_BATCHES=None can ket qua smoke (smoke_timing.json). Chay smoke truoc."
        mb = timing["recommended_max_batches"]
    if timing and mb > 0:
        hours = timing["sec_per_batch"] * mb * EPOCHS / 3600
        print(f"Uoc tinh thoi gian train ~ {hours:.1f} h (ngan sach {TIME_BUDGET_HOURS} h)")
        if hours > TIME_BUDGET_HOURS:
            print("[!] Vuot ngan sach: giam MAX_BATCHES hoac EPOCHS.")
    if mb == 0:
        print("[!] MAX_BATCHES=0: do dai epoch do script quyet dinh; neu timeout hay dat RESUME=True o phien sau.")
    print(f"Run full: epochs={EPOCHS}, max_batches={mb}, batch={BATCH_SIZE}, resume={RESUME}")
    run(build_cmd(EPOCHS, mb, MAX_VAL_BATCHES, CKPT_NAME, resume=RESUME))
else:
    print("Bo qua train that (RUN_MODE='smoke').")

[!] MAX_BATCHES=0: do dai epoch do script quyet dinh; neu timeout hay dat RESUME=True o phien sau.
Run full: epochs=15, max_batches=0, batch=20, resume=False
$ /usr/bin/python3 -c 'import sys, runpy, random
S = 42
random.seed(S)
import numpy as np, torch
np.random.seed(S); torch.manual_seed(S); torch.cuda.manual_seed_all(S)
torch.backends.cudnn.deterministic = True; torch.backends.cudnn.benchmark = False
sys.argv = sys.argv[1:]
runpy.run_path(sys.argv[0], run_name='"'"'__main__'"'"')
' /kaggle/working/LungRay/kaggle/train_resnet50_kaggle.py --backbone resnet50 --epochs 15 --batch-size 20 --lr 0.0001 --weight-decay 0.0001 --head-a-criterion asl --head-c-criterion bce --asl-gamma-neg 4.0 --asl-gamma-pos 1.0 --asl-clip 0.05 --max-batches 0 --max-val-batches 0 --save-name best_resnet50.pth
 LUNGRAY: RESNET-50 MULTI-TASK TRAINING (KAGGLE GPU)
[*] Backbone: RESNET50 | Criterion A: ASL | Epochs: 15 | Seed: 42
[*] Compute Device: cuda (Tesla T4)
[OK] COVIDx CXR-4 manifests: Train=60,816 (pos=5

RuntimeError: Lenh that bai (rc=1). 20 dong cuoi:
[Epoch 15/15] Train Loss: 0.0490 | Val Score: 0.9120 (AUC_A: 0.788, AUC_B: 1.000, AUC_C: 0.989)
[OK] Checkpoint resume artifact synced to /kaggle/working/last_resnet50.pth
[OK] Reloaded best ResNet-50 weights from /kaggle/working/best_resnet50.pth
[Plotting] Training curves exported to /kaggle/working/results/baseline/resnet50_training_curves.png
[OK] Training curves exported to /kaggle/working/results/baseline/resnet50_training_curves.png
[*] Calibrating Youden J Decision Thresholds for ResNet-50...
Traceback (most recent call last):
  File "<string>", line 8, in <module>
    runpy.run_path(sys.argv[0], run_name='__main__')
    ~~~~~~~~~~~~~~^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^
  File "<frozen runpy>", line 294, in run_path
  File "<frozen runpy>", line 98, in _run_module_code
  File "<frozen runpy>", line 88, in _run_code
  File "/kaggle/working/LungRay/kaggle/train_resnet50_kaggle.py", line 200, in <module>
    main()
    ~~~~^^
  File

In [ ]:
# Cell 8: Kiểm định & xuất checkpoint (chỉ sau run full)
def find_newest(name):
    hits = []
    for base in (REPO / "checkpoints", WORK, REPO):
        hits += [p for p in base.rglob(name) if p.is_file()]
    return max(hits, key=lambda p: p.stat().st_mtime) if hits else None

def sha256(p):
    h = hashlib.sha256()
    with open(p, "rb") as f:
        for chunk in iter(lambda: f.read(1 << 20), b""):
            h.update(chunk)
    return h.hexdigest()

if RUN_MODE == "full":
    import inspect
    ckpt_path = find_newest(CKPT_NAME)
    assert ckpt_path is not None, f"Khong tim thay {CKPT_NAME} sau khi train."
    print("Checkpoint:", ckpt_path, f"({ckpt_path.stat().st_size / 1e6:.1f} MB)")

    ckpt = torch.load(ckpt_path, map_location="cpu", weights_only=False)  # file tu sinh ra, tin cay
    if isinstance(ckpt, dict) and any(isinstance(v, dict) for v in ckpt.values()):
        print("Top-level keys:", list(ckpt.keys()))
    sd = ckpt
    if isinstance(ckpt, dict):
        for k in ("model_state_dict", "state_dict", "model"):
            if k in ckpt and isinstance(ckpt[k], dict):
                sd = ckpt[k]
                break
    sd = {k.replace("module.", "", 1): v for k, v in sd.items()}

    shapes = {}
    for h in ("head_a", "head_b", "head_c"):
        ks = [k for k in sd if k.startswith(h) and k.endswith("weight")]
        assert ks, f"Thieu khoa {h}.* trong state_dict"
        shapes[h] = tuple(sd[ks[0]].shape)
    print("Head weight shapes:", shapes)
    assert shapes["head_a"] == (14, 2048), shapes["head_a"]
    assert shapes["head_b"] == (1, 2048), shapes["head_b"]
    assert shapes["head_c"][1] == 2048 and shapes["head_c"][0] in (1, 3), shapes["head_c"]
    ln_keys = [k for k in sd if k.startswith(("norm.", "head_norm"))]
    print("Khoa LayerNorm cap cao (ky vong: 0 vi parameter-free):", ln_keys)
    if shapes["head_c"][0] != 1:
        print("[!] Head C khong phai nhi phan (1 logit) trong khi script train theo BCE nhi phan: kiem tra spec.")

    from src.models.multi_task_net import MultiTaskChestXrayNet
    params = inspect.signature(MultiTaskChestXrayNet.__init__).parameters
    kw = {}
    if "backbone_name" in params: kw["backbone_name"] = BACKBONE
    if "pretrained" in params: kw["pretrained"] = False
    if "num_classes_c" in params: kw["num_classes_c"] = shapes["head_c"][0]
    if "head_norm" in params: kw["head_norm"] = "layernorm"
    model = MultiTaskChestXrayNet(**kw)
    model.load_state_dict(sd, strict=True)          # loi neu thieu/thua khoa
    model.eval()
    n_params = sum(p.numel() for p in model.parameters())
    print(f"Tham so: {n_params / 1e6:.2f} M")
    assert 23.0e6 < n_params < 24.2e6, "So tham so khong khop ResNet-50 + 3 head"

    with torch.no_grad():
        out = model(torch.randn(1, 3, 512, 512))
    tensors = []
    def collect(o):
        if torch.is_tensor(o): tensors.append(o)
        elif isinstance(o, dict): [collect(v) for v in o.values()]
        elif isinstance(o, (list, tuple)): [collect(v) for v in o]
    collect(out)
    assert tensors and all(torch.isfinite(t).all() for t in tensors), "Output co NaN/Inf"
    print("Forward OK, shapes:", [tuple(t.shape) for t in tensors])

    final_full = WORK / CKPT_NAME
    if ckpt_path.resolve() != final_full.resolve():
        import shutil
        shutil.copy(ckpt_path, final_full)
    slim = WORK / "best_resnet50_weights_only.pth"
    tmp = slim.with_suffix(".tmp")
    torch.save(sd, tmp)
    os.replace(tmp, slim)
    for p in (final_full, slim):
        print(f"{p.name}: {p.stat().st_size / 1e6:.1f} MB | sha256 {sha256(p)[:16]}...")
    if THRESHOLDS_SRC.exists():
        import shutil
        shutil.copy(THRESHOLDS_SRC, THRESHOLDS_OUT)
        print("Nguong:", THRESHOLDS_OUT, "(rieng ResNet-50, khong ghi de nguong DenseNet)")
    else:
        print(f"[!] Khong thay {THRESHOLDS_SRC}: calibrate co the da that bai, xem log train.")
    print("Luu y: weights-only ~94 MB; neu file goc ~280 MB thi no chua ca optimizer.")
    print("Kiem tra app/backend/services.py doc dinh dang nao truoc khi hot-reload.")
else:
    print("Bo qua kiem dinh (RUN_MODE='smoke').")

In [ ]:
# Cell 9: Tổng kết
for p in sorted(WORK.glob("*")):
    if p.is_file():
        print(f"{p.name:45s} {p.stat().st_size / 1e6:8.1f} MB")
print("\nViec tiep theo: (1) tai file ve; (2) ghep dung ngưỡng optimal_thresholds_resnet50.json "
      "voi checkpoint nay; (3) danh gia test dung 1 lan; (4) hot-reload.")